# 00 -- Benchmark Exploration

Explore the cognitive bias benchmark: 284 items across 7 categories, each with
a conflict (S1-lure) variant and a matched no-conflict control.

**What this notebook does:**
1. Load and validate the benchmark JSONL
2. Category breakdown (bar chart)
3. Conflict vs control balance (pie chart)
4. Example matched pairs
5. Difficulty distribution
6. Paraphrase statistics
7. Prompt length distribution

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Ensure the package is importable
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.benchmark import (
    load_benchmark,
    validate_benchmark,
    iter_matched_pairs,
    filter_by_category,
    filter_conflict,
    group_by_matched_pair,
)
from s1s2.viz.theme import set_paper_theme, COLORS

set_paper_theme()

In [ ]:
# --- Load benchmark ---
BENCHMARK_PATH = REPO_ROOT / "data" / "benchmark" / "benchmark.jsonl"

items = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(items)} items from {BENCHMARK_PATH.name}")

# Run full validation
report = validate_benchmark(items)
print(f"Validation passed: {report.passed}")
if report.errors:
    for e in report.errors[:5]:
        print(f"  ERROR: {e}")
if report.warnings:
    for w in report.warnings[:5]:
        print(f"  WARN: {w}")

## Category Breakdown

In [ ]:
cats = Counter(it.category for it in items)
cats_sorted = sorted(cats.items(), key=lambda x: x[1], reverse=True)
cat_names, cat_counts = zip(*cats_sorted)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(cat_names, cat_counts, color=COLORS["standard"], edgecolor="white")
ax.set_xlabel("Number of items")
ax.set_title("Benchmark items by category")
for bar, count in zip(bars, cat_counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTotal categories: {len(cats)}")
for name, count in cats_sorted:
    print(f"  {name}: {count} ({count / len(items):.1%})")

## Conflict vs Control Balance

In [ ]:
n_conflict = sum(1 for it in items if it.conflict)
n_control = len(items) - n_conflict

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Pie chart
axes[0].pie(
    [n_conflict, n_control],
    labels=[f"Conflict (S1-lure)\nn={n_conflict}", f"No-conflict (control)\nn={n_control}"],
    colors=[COLORS["s1"], COLORS["s2"]],
    autopct="%1.0f%%",
    startangle=90,
)
axes[0].set_title("Conflict vs Control")

# Per-category breakdown
conflict_per_cat = Counter(it.category for it in items if it.conflict)
control_per_cat = Counter(it.category for it in items if not it.conflict)
all_cats = sorted(set(conflict_per_cat) | set(control_per_cat))
x = np.arange(len(all_cats))
w = 0.35
axes[1].bar(x - w / 2, [conflict_per_cat[c] for c in all_cats], w,
            label="Conflict", color=COLORS["s1"])
axes[1].bar(x + w / 2, [control_per_cat[c] for c in all_cats], w,
            label="Control", color=COLORS["s2"])
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_cats, rotation=45, ha="right")
axes[1].set_ylabel("Count")
axes[1].set_title("Conflict/Control per category")
axes[1].legend()

plt.tight_layout()
plt.show()

## Example Matched Pairs

Show 3 representative conflict/control pairs to illustrate the benchmark structure.

In [ ]:
pairs = list(iter_matched_pairs(items))
print(f"Total matched pairs: {len(pairs)}\n")

# Show 3 pairs from different categories
shown_cats = set()
shown = 0
for conflict_item, control_item in pairs:
    if conflict_item.category in shown_cats:
        continue
    shown_cats.add(conflict_item.category)
    print(f"{'='*70}")
    print(f"Pair: {conflict_item.matched_pair_id}  |  Category: {conflict_item.category}")
    print(f"{'='*70}")
    print(f"\n  CONFLICT (S1-lure present):")
    print(f"  ID: {conflict_item.id}")
    print(f"  Prompt: {conflict_item.prompt[:200]}..." if len(conflict_item.prompt) > 200
          else f"  Prompt: {conflict_item.prompt}")
    print(f"  Correct: {conflict_item.correct_answer}  |  Lure: {conflict_item.lure_answer}")
    print(f"\n  CONTROL (no S1-lure):")
    print(f"  ID: {control_item.id}")
    print(f"  Prompt: {control_item.prompt[:200]}..." if len(control_item.prompt) > 200
          else f"  Prompt: {control_item.prompt}")
    print(f"  Correct: {control_item.correct_answer}")
    print()
    shown += 1
    if shown >= 3:
        break

## Difficulty Distribution

In [ ]:
difficulties = [it.difficulty for it in items]
diff_counter = Counter(difficulties)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Overall histogram
levels = sorted(diff_counter.keys())
axes[0].bar(levels, [diff_counter[d] for d in levels],
            color=COLORS["standard"], edgecolor="white")
axes[0].set_xlabel("Difficulty (1-5)")
axes[0].set_ylabel("Count")
axes[0].set_title("Overall difficulty distribution")
axes[0].set_xticks(levels)

# Per-category difficulty as box plot
df = pd.DataFrame([{"category": it.category, "difficulty": it.difficulty} for it in items])
df.boxplot(column="difficulty", by="category", ax=axes[1], grid=False)
axes[1].set_title("Difficulty by category")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Difficulty")
plt.suptitle("")  # Remove default boxplot title
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Difficulty x conflict
print("\nDifficulty by conflict status:")
conflict_diffs = [it.difficulty for it in items if it.conflict]
control_diffs = [it.difficulty for it in items if not it.conflict]
print(f"  Conflict: mean={np.mean(conflict_diffs):.2f}, median={np.median(conflict_diffs):.1f}")
print(f"  Control:  mean={np.mean(control_diffs):.2f}, median={np.median(control_diffs):.1f}")

## Paraphrase Statistics

In [ ]:
paraphrase_counts = [len(it.paraphrases) for it in items]
has_paraphrases = sum(1 for c in paraphrase_counts if c > 0)
total_paraphrases = sum(paraphrase_counts)

print(f"Items with paraphrases: {has_paraphrases}/{len(items)} ({has_paraphrases/len(items):.1%})")
print(f"Total paraphrase variants: {total_paraphrases}")
if total_paraphrases > 0:
    print(f"Mean paraphrases per item (among those with any): "
          f"{total_paraphrases / max(has_paraphrases, 1):.1f}")

# Distribution of paraphrase counts
pc = Counter(paraphrase_counts)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(sorted(pc.keys()), [pc[k] for k in sorted(pc.keys())],
       color=COLORS["standard"], edgecolor="white")
ax.set_xlabel("Number of paraphrases")
ax.set_ylabel("Number of items")
ax.set_title("Paraphrase count distribution")
plt.tight_layout()
plt.show()

## Prompt Length Distribution

In [ ]:
prompt_lengths = [len(it.prompt) for it in items]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(prompt_lengths, bins=30, color=COLORS["standard"], edgecolor="white", alpha=0.8)
ax.axvline(np.median(prompt_lengths), color=COLORS["s1"], linestyle="--",
           label=f"Median = {np.median(prompt_lengths):.0f} chars")
ax.set_xlabel("Prompt length (characters)")
ax.set_ylabel("Count")
ax.set_title("Prompt length distribution")
ax.legend()
plt.tight_layout()
plt.show()

# Source breakdown
sources = Counter(it.source for it in items)
print("\nSources:")
for src, count in sources.most_common():
    print(f"  {src}: {count} ({count / len(items):.1%})")

## Summary

**Key takeaways:**
- Benchmark contains matched conflict/control pairs across 7 cognitive bias categories
- Each conflict item has a surface-matched no-conflict control, controlling for task difficulty
- Difficulty ratings span 1-5; verify matched pairs have identical difficulty
- Paraphrases provide robustness checks against prompt-specific artifacts

**Next steps:** Run model extraction (`scripts/extract.py`) to generate activations, then proceed to notebooks 01-05.